In [1]:
import pandas as pd 


In [2]:
dmg_t = pd.read_hdf("~/Ensi/project/oocytes-macaque/allcools/allcools-subsampled/alloocytes/results-pooled/L1_DMG_CGN_ppoledoocytes.hdf")


In [3]:
dmgs = pd.read_csv("../pooledoocytes/mCG_genenames_coords_withlifted.csv")
dmgs.columns = ("Gene", "chrom", "start", "end", "Gene_name")
dmgs

,Gene,chrom,start,end,Gene_name
0,ENSMMUG00000054615,chr1,101331787,101333472,NaN
1,ENSMMUG00000053737,chr1,133742526,133752842,BARHL2
2,ENSMMUG00000063102,chr1,204242900,204245913,CAMK2N1
3,ENSMMUG00000016814,chr10,43467723,43471378,THBD
4,ENSMMUG00000010410,chr10,43927729,43933713,FOXA2
...,...,...,...,...,...
69,ENSMMUG00000030511,chr9,97765488,97766328,FRAT1
70,ENSMMUG00000054006,chr9,124894605,124899311,NKX1-2
71,ENSMMUG00000065350,chr9,133693821,133695328,UTF1
72,ENSMMUG00000003201,chrX,38619175,38624258,MID1IP1


In [6]:
dmg_table = dmg_t.merge(dmgs, on = "Gene", how="inner")
dmg_table.to_csv("dmgs_CGN_tableWithdiresctionandStats.csv")
dmg_table

,Cluster,Gene,pval,adjpval,logfc,Direction,chrom,start,end,Gene_name
0,c0,ENSMMUG00000004541,0.000315,0.668065,1.964730,hyper methylation,chr11,53556785,53558414,HOXC12
1,c0,ENSMMUG00000064061,0.000420,0.668065,0.916519,hyper methylation,chr15,1606923,1611412,NaN
2,c0,ENSMMUG00000047902,0.001365,0.668065,1.479827,hyper methylation,chr15,1017481,1018839,SSNA1
3,c0,ENSMMUG00000061125,0.001365,0.668065,-1.217649,hypo methylation,chr6,20789023,20792440,"MT-ND5, MT-ND6, MT-TE, MT-CYB"
4,c0,ENSMMUG00000044556,0.001622,0.668065,1.044826,hyper methylation,chr11,7037079,7038774,SPSB2
...,...,...,...,...,...,...,...,...,...,...
69,c0,ENSMMUG00000057382,0.046730,0.757343,0.961417,hyper methylation,chr12,12267455,12272419,EN1
70,c0,ENSMMUG00000013458,0.048128,0.757343,0.802934,hyper methylation,chr3,87947758,87951764,HOXA7
71,c0,ENSMMUG00000064772,0.048128,0.757343,1.708590,hyper methylation,chr12,106414852,106418243,FEV
72,c0,ENSMMUG00000064376,0.048128,0.757343,0.993567,hyper methylation,chr11,107452569,107453091,ASCL4


In [31]:
from statsmodels.stats.multitest import multipletests
import pandas as pd

# Your confirmed TF symbols from earlier
tf_genes = {
    'HOXC12', 'GSX1', 'FEV', 'SOX18', 'FOXA2', 'FOXQ1', 'HOXB13', 'BARHL2',
    'CITED2', 'HOXD12', 'UTF1', 'MYC', 'HOXD1', 'PURB', 'PHF10', 'LHX3',
    'DLX3', 'LHX5', 'ASCL4', 'EN1', 'TCF19', 'ZNF174', 'ZIC3', 'ZIC1',
    'FLI1', 'CDX2', 'NKX1-2', 'HOXA7', 'VGLL2'
}

def apply_familywise_fdr(dmg_table, tf_genes, tf_threshold=0.10, other_threshold=0.20):
    dmg_table = dmg_table.copy()

    # Annotate TF vs other (works on gene symbols OR ensembl IDs if tf_genes contains them)
    dmg_table['is_tf'] = dmg_table['Gene_name'].isin(tf_genes)
    dmg_table['family'] = dmg_table['is_tf'].map({True: 'TF', False: 'Other'})

    print("=== Family & Direction breakdown ===")
    print(dmg_table.groupby(['family', 'Direction']).size().to_string())
    print()

    results = []
    for (is_tf, direction), group in dmg_table.groupby(['is_tf', 'Direction']):
        family_label = 'TF' if is_tf else 'Other'
        threshold = tf_threshold if is_tf else other_threshold
        n = len(group)

        # FDR correction within this family+direction group
        _, fdr_corrected, _, _ = multipletests(group['pval'], method='fdr_bh')

        group = group.copy()
        group['fdr']           = fdr_corrected
        group['fdr_threshold'] = threshold
        group['passes_fdr']    = fdr_corrected < threshold

        n_pass = group['passes_fdr'].sum()
        print(f"[{family_label} | {direction}] n={n}, threshold={threshold}, passing={n_pass}")
        results.append(group)

    final = pd.concat(results).sort_values(['family', 'Direction', 'fdr'])

    print("\n=== Summary ===")
    summary = final.groupby(['family', 'Direction']).agg(
        total   = ('Gene_name', 'count'),
        passing = ('passes_fdr', 'sum')
    )
    print(summary.to_string())

    return final


# Run
dmg_fdr = apply_familywise_fdr(dmg_table, tf_genes)

# View only passing genes
passing = dmg_fdr[dmg_fdr['passes_fdr']]
print(f"\nTotal passing genes: {len(passing)}")
print(passing[['Cluster', 'Gene', 'family', 'Direction', 'logfc', 'pval', 'fdr', 'fdr_threshold']])

=== Family & Direction breakdown ===
family  Direction        
Other   hyper methylation    40
        hypo methylation      5
TF      hyper methylation    28
        hypo methylation      1

[Other | hyper methylation] n=40, threshold=0.2, passing=40
[Other | hypo methylation] n=5, threshold=0.2, passing=5
[TF | hyper methylation] n=28, threshold=0.1, passing=28
[TF | hypo methylation] n=1, threshold=0.1, passing=1

=== Summary ===
                          total  passing
family Direction                        
Other  hyper methylation     34       40
       hypo methylation       4        5
TF     hyper methylation     28       28
       hypo methylation       1        1

Total passing genes: 74
   Cluster                Gene family          Direction     logfc      pval  \
1       c0  ENSMMUG00000064061  Other  hyper methylation  0.916519  0.000420   
2       c0  ENSMMUG00000047902  Other  hyper methylation  1.479827  0.001365   
4       c0  ENSMMUG00000044556  Other  hyper methyla

In [32]:
passing.to_csv("family-direction-wise-fdr.csv", sep="\t")

In [33]:
passing = pd.read_csv("family-direction-wise-fdr.csv", sep="\t")
passing

,Unnamed: 0,Cluster,Gene,pval,adjpval,logfc,Direction,chrom,start,end,Gene_name,is_tf,family,fdr,fdr_threshold,passes_fdr
0,1,c0,ENSMMUG00000064061,0.000420,0.668065,0.916519,hyper methylation,chr15,1606923,1611412,NaN,False,Other,0.016799,0.2,True
1,2,c0,ENSMMUG00000047902,0.001365,0.668065,1.479827,hyper methylation,chr15,1017481,1018839,SSNA1,False,Other,0.017888,0.2,True
2,4,c0,ENSMMUG00000044556,0.001622,0.668065,1.044826,hyper methylation,chr11,7037079,7038774,SPSB2,False,Other,0.017888,0.2,True
3,5,c0,ENSMMUG00000039505,0.001923,0.668065,0.829948,hyper methylation,chr7,165204235,165213153,AMN,False,Other,0.017888,0.2,True
4,7,c0,ENSMMUG00000023647,0.002575,0.668065,1.430972,hyper methylation,chr9,22557864,22560588,SKIDA1,False,Other,0.017888,0.2,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,69,c0,ENSMMUG00000057382,0.046730,0.757343,0.961417,hyper methylation,chr12,12267455,12272419,EN1,True,TF,0.048128,0.1,True
70,70,c0,ENSMMUG00000013458,0.048128,0.757343,0.802934,hyper methylation,chr3,87947758,87951764,HOXA7,True,TF,0.048128,0.1,True
71,71,c0,ENSMMUG00000064772,0.048128,0.757343,1.708590,hyper methylation,chr12,106414852,106418243,FEV,True,TF,0.048128,0.1,True
72,72,c0,ENSMMUG00000064376,0.048128,0.757343,0.993567,hyper methylation,chr11,107452569,107453091,ASCL4,True,TF,0.048128,0.1,True


In [ ]:
for (is_tf, direction), group in dmg_table.groupby(['is_tf', 'Direction']):
```

This creates **4 groups**:

| Group | `is_tf` | `Direction` |
|-------|---------|-------------|
| 1 | `True` | `hyper methylation` |
| 2 | `True` | `hypo methylation` |
| 3 | `False` | `hyper methylation` |
| 4 | `False` | `hypo methylation` |

FDR correction via `multipletests` is applied **independently within each of these 4 groups**, so genes are only competing against others in the same family+direction bucket. 